# 06 · Compression

**Goals**
- Make the trained model smaller (and faster where possible) without losing segmentation quality
- For every variant: PQ, weight-file size, network time per image

**Methods**

| Method | What it does | Smaller file? | Faster? |
|---|---|---|---|
| fp16 weights | store and run every weight as a 16-bit instead of a 32-bit number | yes, half | yes on GPU |
| Global L1 pruning | set the smallest x% of weights to zero (first and last layer left alone) | only once compressed, since zeros compress well | no: a GPU multiplies zeros as fast as other numbers |
| Pruning + fp16 | both | yes | yes on GPU |

Pruning is applied without retraining, so this measures how many weights the trained model can lose as it is.

**Evaluation:** the final pipeline of notebook 07 (same post-processing, same TTA) on the fold-0 validation images,
scored against the competition's ground truth (pycocotools masks). On Kaggle, attach the competition data, the
preprocessed-images dataset and the outputs of notebooks 04 and 07 (or 05); turn on GPU and Internet.
`COS711_SMOKE=1` checks the pipeline on a few images on CPU.

In [ ]:
# --- Setup: works both locally and on Kaggle ---
import os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    # Pull the shared src/ code from GitHub (requires Internet ON in notebook settings)
    if not Path("/kaggle/working/COS711-A2").exists():
        !git clone -q https://github.com/AryanMohanlall/COS711-A2.git /kaggle/working/COS711-A2
    ROOT = Path("/kaggle/working/COS711-A2")
elif Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

sys.path.insert(0, str(ROOT))

from src.data import get_data_dir
DATA_DIR = get_data_dir()
print("Kaggle:", IN_KAGGLE, "| data dir:", DATA_DIR)

try:
    import segmentation_models_pytorch
except ImportError:
    %pip install -q segmentation-models-pytorch
try:
    import pycocotools
except ImportError:
    %pip install -q pycocotools

In [ ]:
import gzip, json, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.nn.utils import prune
from tqdm.auto import tqdm
from pycocotools import mask as mask_utils

from src.data import ANNOTATION_FILE, load_annotations
from src.preprocess import load_preprocessed, disk_mask
from src.augment import NO_AUG
from src.dataset import FilamentDataset
from src.model import build_model, model_size
from src.train import make_loader
from src.postprocess import components, drop_small, upsample
from src.metrics import match_instances, pq_from_counts

SMOKE = os.environ.get("COS711_SMOKE") == "1"   # tiny CPU run: checks the pipeline, numbers meaningless
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

S = 1024              # working resolution (as notebook 04)
FULL = 2048           # image resolution
LIMB_CUT = 0.985      # pixels beyond 0.985 R are never filament
VAL_FOLD = 0
PRUNING_LEVELS = (0.2, 0.4, 0.6, 0.8)   # share of weights set to zero

if SMOKE:
    OUT_DIR = ROOT / "outputs" / "smoke"
    BATCH = 1
    WORKERS = 0
else:
    OUT_DIR = ROOT
    BATCH = 8
    WORKERS = min(4, os.cpu_count())

FIG_DIR = OUT_DIR / "figures" / "06"
ART_OUT = OUT_DIR / "artifacts"
WEIGHTS_DIR = OUT_DIR / "outputs" / "compressed_weights"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ART_OUT.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 4)


def previous_output(rel, required=True):
    """A file written by an earlier notebook: an attached notebook output on Kaggle if there is one, else the repo copy."""
    candidates = []
    if IN_KAGGLE:
        candidates = sorted(Path("/kaggle/input").glob(f"**/{rel}"))
    candidates.append(OUT_DIR / rel)
    candidates.append(ROOT / rel)
    for path in candidates:
        if path.exists():
            print(rel, "←", path)
            return path
    if not required:
        return None
    message = f"{rel} not found: run the earlier notebooks first"
    if IN_KAGGLE:
        message += " and attach their outputs (Add Input → Notebook Output Files)"
    raise FileNotFoundError(message)


def preprocessed_dir(probe):
    """Folder that holds full_<S>/ with the preprocessed .npy files of notebook 02."""
    if IN_KAGGLE:
        candidates = []
        for f in sorted(Path("/kaggle/input").glob(f"**/{probe}")):
            candidates.append(f.parent)
    else:
        candidates = [ROOT / "cache" / f"full_{S}"]
    found = None
    for folder in candidates:
        if (folder / probe).exists():
            found = folder
            break
    if found is None:
        raise FileNotFoundError(f"preprocessed images ({probe}) not found: run notebook 02 / attach the dataset")
    print(f"cache/full_{S}", "←", found)
    # load_preprocessed reads <cache_dir>/full_<S>/<obs>.npy, so a folder with another name is linked under /tmp
    if found.name != f"full_{S}":
        link = Path("/tmp/cos711_cache") / f"full_{S}"
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            link.symlink_to(found, target_is_directory=True)
        found = link
    return found.parent


IMAGE_STATS_CSV = previous_output("artifacts/image_stats.csv")            # notebook 01
EXCLUDED_CSV = previous_output("artifacts/excluded_annotations.csv")      # notebook 02
FOLDS_CSV = previous_output("artifacts/folds.csv")                        # notebook 02
first_obs = pd.read_csv(FOLDS_CSV).obs.iloc[0]
CACHE_DIR = preprocessed_dir(f"{first_obs}.npy")                         # notebook 02

# The final pipeline: notebook 07's settings if attached, otherwise notebook 05's
settings_file = previous_output("artifacts/inference.json", required=False)   # notebook 07
if settings_file is None:
    settings_file = previous_output("artifacts/postprocess.json")            # notebook 05
with open(settings_file) as fh:
    chosen = json.load(fh)
MODEL_NAME = chosen["model"]
USE_TTA = chosen["tta"]
SHIFT = 0.0
if "shift" in chosen:
    SHIFT = chosen["shift"]
SETTINGS = {"high": chosen["high"], "low": chosen["low"], "min_area": chosen["min_area"],
            "gap": chosen["gap"], "grow": chosen["grow"]}
CKPT = previous_output(f"checkpoints/{MODEL_NAME}.pt")                     # notebook 04

print()
print("model:", MODEL_NAME, "| TTA:", USE_TTA, "| shift:", SHIFT, "| post-processing:", SETTINGS)
if DEVICE.type == "cuda":
    print("device:", DEVICE, torch.cuda.get_device_name(), "| smoke:", SMOKE)
else:
    print("device:", DEVICE, "| smoke:", SMOKE)

## 1. Validation data and scoring

The fold-0 validation images, the input normalisation of notebook 04 (same code, same images, so the same values),
and the competition's ground truth: every annotation rasterised with pycocotools (notebook 07, section 1).

In [ ]:
images, anns = load_annotations(DATA_DIR)
excluded = pd.read_csv(EXCLUDED_CSV)
anns = anns[~anns.id.isin(excluded.ann_id)].reset_index(drop=True)
folds = pd.read_csv(FOLDS_CSV)
geo_full = pd.read_csv(IMAGE_STATS_CSV).set_index("obs")[["cx", "cy", "radius"]]
paths = images.drop_duplicates("obs").set_index("obs").path
if SMOKE:
    folds = folds.groupby("fold").head(2).reset_index(drop=True)

train_obs = folds.loc[folds.fold != VAL_FOLD, "obs"].tolist()
val_obs = folds.loc[folds.fold == VAL_FOLD, "obs"].tolist()

# Input normalisation exactly as in notebook 04
probe = FilamentDataset(train_obs, images, anns, geo_full, size=S, cache_dir=CACHE_DIR)
step = max(1, len(probe.obs) // 100)
values = []
for o in probe.obs[::step]:
    cx, cy, r = geo_full.loc[o]
    x = load_preprocessed(paths[o], (cx, cy, r), S, cache_dir=CACHE_DIR)
    on_disk_pixels = x[disk_mask(x.shape, cx * S / FULL, cy * S / FULL, r * S / FULL, LIMB_CUT)]
    values.append(on_disk_pixels[::7])
values = np.concatenate(values)
med = float(np.median(values))
NORM = (med, float(1.4826 * np.median(np.abs(values - med))))
print("normalisation (centre, scale):", (round(NORM[0], 4), round(NORM[1], 4)))

val_ds = FilamentDataset(val_obs, images, anns, geo_full, size=S, norm=NORM, aug=NO_AUG, cache_dir=CACHE_DIR)
val_loader = make_loader(val_ds, BATCH, train=False, workers=WORKERS)

# Competition ground truth, built once: one pycocotools instance map per annotation set
with open(DATA_DIR / "train" / ANNOTATION_FILE) as fh:
    raw = json.load(fh)
segmentation = {}
for a in raw["annotations"]:
    segmentation[a["id"]] = a["segmentation"]

ground_truth = {}
for o in tqdm(val_ds.obs, desc="ground truth"):
    maps = []
    for image_id in val_ds.set_ids[o]:
        m = np.zeros((FULL, FULL), np.int16)
        k = 0
        for ann_id in anns.loc[anns.image_id == image_id, "id"]:   # later annotations win overlaps
            k += 1
            rles = mask_utils.frPyObjects(segmentation[ann_id], FULL, FULL)
            m[mask_utils.decode(mask_utils.merge(rles)) > 0] = k
        maps.append(m)
    ground_truth[o] = maps
print(len(ground_truth), "validation images")

In [ ]:
def disk_kernel(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def instances(p, on_disk, settings):
    """Instance map (0 = background) from a full-resolution probability map: the post-processing of notebook 05."""
    high = settings["high"]
    low = settings["low"]
    if low is None:
        fg = (p > high) & on_disk
    else:
        fg = (p > low) & on_disk
    lab, _ = components(fg)
    if low is not None:
        # Hysteresis: keep a component of p > low only if it contains a pixel with p > high
        keep = np.zeros(lab.max() + 1, bool)
        keep[np.unique(lab[fg & (p > high)])] = True
        keep[0] = False
        fg = keep[lab]
    if settings["gap"] > 0:
        # Pieces closer than `gap` px form one instance; the pixels stay the predicted ones
        dilated = cv2.dilate(fg.astype(np.uint8), disk_kernel(settings["gap"])) > 0
        groups, _ = components(dilated)
        lab = np.where(fg, groups, 0)
    elif low is not None:
        lab, _ = components(fg)
    lab = drop_small(lab, np.bincount(lab.ravel()), settings["min_area"])
    if settings["grow"] > 0:
        grown = cv2.dilate(lab.astype(np.uint16), disk_kernel(settings["grow"])).astype(lab.dtype)
        lab = np.where((lab == 0) & on_disk, grown, lab)
    return lab


def score(probs):
    """Pooled PQ / SQ / RQ of the final pipeline against the competition's ground truth."""
    c = np.zeros(4)
    for o in probs:
        p = upsample(probs[o], FULL)
        if SHIFT:
            M = np.float32([[1, 0, -SHIFT], [0, 1, -SHIFT]])
            p = cv2.warpAffine(p, M, (FULL, FULL), flags=cv2.INTER_LINEAR)
        cx, cy, r = val_ds.geo[o]
        pred = instances(p, disk_mask((FULL, FULL), cx, cy, r, LIMB_CUT), SETTINGS)
        for gt in ground_truth[o]:
            m = match_instances(gt, pred)
            c += (m["tp"], m["fp"], m["fn"], m["iou_sum"])
    return pq_from_counts(int(c[0]), int(c[1]), int(c[2]), c[3])

## 2. Model variants

In [ ]:
def load_model():
    m = build_model(weights=None).to(DEVICE)
    m.load_state_dict(torch.load(CKPT, map_location=DEVICE))
    return m.eval()


def prunable_layers(model):
    """Every convolution weight except the first one (reads the image) and the last one (writes the mask):
    those two are tiny and the most sensitive, so pruning them costs quality for almost no saving."""
    convs = []
    for module in model.modules():
        if isinstance(module, torch.nn.Conv2d):
            convs.append(module)
    layers = []
    for module in convs[1:-1]:
        layers.append((module, "weight"))
    return layers


def sparsity(model):
    """Share of all model weights that are exactly zero."""
    total = 0
    zeros = 0
    for p in model.parameters():
        total += p.numel()
        zeros += int((p == 0).sum())
    return zeros / total


def file_sizes(model, name):
    """Size (MB) of the saved weight file, and of the same file compressed with gzip."""
    path = WEIGHTS_DIR / f"{name}.pt"
    torch.save(model.state_dict(), path)
    raw_bytes = path.read_bytes()
    compressed = gzip.compress(raw_bytes, compresslevel=6)
    return len(raw_bytes) / 2 ** 20, len(compressed) / 2 ** 20


def predict_and_time(model, half):
    """Probability map of every validation image, and the network time per image (ms).
    With TTA (if the final pipeline uses it) averaged over the 8 rotations / flips."""
    run_half = half and DEVICE.type == "cuda"   # fp16 convolutions only run on the GPU
    run_model = model
    if half and not run_half:
        run_model = model.float()                # CPU: size is measured in fp16, inference runs in fp32
    use_autocast = DEVICE.type == "cuda" and not run_half   # fp32 weights: mixed precision, as in notebook 07
    out = {}
    network_s = 0.0
    with torch.no_grad():
        for x, _, idx in tqdm(val_loader, desc="predict", leave=False):
            x = x.to(DEVICE)
            if run_half:
                x = x.half()
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            if USE_TTA:
                views = []
                for k in range(4):
                    for flip in (False, True):
                        views.append((k, flip))
            else:
                views = [(0, False)]
            total = 0
            for k, flip in views:
                view = torch.rot90(x, k, (2, 3))
                if flip:
                    view = view.flip(3)
                with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_autocast):
                    p = torch.sigmoid(run_model(view).float())
                if flip:
                    p = p.flip(3)
                total = total + torch.rot90(p, -k, (2, 3))
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            network_s += time.perf_counter() - t0
            prob = (total / len(views))[:, 0].cpu().numpy()
            ids = idx.tolist()
            for i in range(len(ids)):
                out[val_ds.obs[ids[i]]] = prob[i].astype(np.float16)
    return out, 1000 * network_s / len(out)


# (name, share of weights pruned, fp16 weights)
VARIANTS = [("baseline (fp32 weights)", 0.0, False), ("fp16 weights", 0.0, True)]
for amount in PRUNING_LEVELS:
    VARIANTS.append((f"pruned {int(amount * 100)}%", amount, False))
VARIANTS.append(("pruned 40% + fp16", 0.4, True))

In [ ]:
rows = []
for name, amount, half in VARIANTS:
    print(name)
    model = load_model()
    if amount > 0:
        # Global L1 pruning: the `amount` smallest weights (by absolute value) across all layers become zero
        prune.global_unstructured(prunable_layers(model), pruning_method=prune.L1Unstructured, amount=amount)
        for module, weight_name in prunable_layers(model):
            prune.remove(module, weight_name)   # make the zeros permanent
    if half:
        model = model.half()

    file_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct").replace("+", "and")
    raw_mb, gzip_mb = file_sizes(model, file_name)
    probs, network_ms = predict_and_time(model, half)
    result = score(probs)
    rows.append({"variant": name, "zero weights": sparsity(model), "file MB": raw_mb, "gzip MB": gzip_mb,
                 "network ms / image": network_ms, "pq": result["pq"], "sq": result["sq"], "rq": result["rq"]})
    print("  PQ", round(result["pq"], 4), "| file", round(raw_mb, 1), "MB | gzip", round(gzip_mb, 1), "MB |",
          round(network_ms, 1), "ms / image")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame(rows).set_index("variant")
base_pq = results.loc["baseline (fp32 weights)", "pq"]
base_gzip = results.loc["baseline (fp32 weights)", "gzip MB"]
results["PQ change"] = results.pq - base_pq
results["gzip size vs baseline"] = results["gzip MB"] / base_gzip
results.to_csv(ART_OUT / "compression_results.csv")
results.round(4)

## 3. Quality vs size

In [ ]:
pruned = results[results.index.str.startswith("pruned") & ~results.index.str.contains("fp16")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(100 * pruned["zero weights"], pruned.pq, marker="o", label="pruned (fp32)")
axes[0].axhline(base_pq, color="grey", linestyle="--", label="baseline")
axes[0].scatter([100 * results.loc["fp16 weights", "zero weights"]], [results.loc["fp16 weights", "pq"]],
                color="tab:red", zorder=3, label="fp16 weights")
axes[0].set(xlabel="weights set to zero (%)", ylabel="validation PQ", title="Quality after pruning (no retraining)")
axes[0].legend(fontsize=8)

axes[1].barh(results.index, results["file MB"], color="lightgrey", label="weight file")
axes[1].barh(results.index, results["gzip MB"], color="tab:blue", label="gzip-compressed")
axes[1].invert_yaxis()
axes[1].set(xlabel="MB", title="Model size")
axes[1].legend(fontsize=8)
plt.tight_layout()
savefig("compression")
plt.show()

## Summary

In [ ]:
# Smallest compressed model that keeps PQ within 0.005 of the baseline
good = results[results["PQ change"] >= -0.005]
best = good.sort_values("gzip MB").index[0]
params = model_size(load_model())["params_M"]

print(f"Baseline: {round(params, 2)} M parameters, {round(results.loc['baseline (fp32 weights)', 'file MB'], 1)} MB, "
      f"PQ {round(base_pq, 4)}")
print(f"Smallest within 0.005 PQ: {best}: {round(results.loc[best, 'gzip MB'], 1)} MB compressed "
      f"({round(100 * results.loc[best, 'gzip size vs baseline'])}% of the baseline), "
      f"PQ {round(results.loc[best, 'pq'], 4)}")
print("Pruned weights are still stored and multiplied as dense numbers: the speed only changes with fp16.")
results[["zero weights", "gzip MB", "network ms / image", "pq", "PQ change"]].round(4)